# 07 · Reward-function comparison  *(branch `exp/07-reward-comparison`)*

Brief §3.4 (d). Same agent and state (D), different rewards, 5 seeds each:

| reward | |
|---|---|
| `net_asset_change` | change in net assets, net of costs (the notebook's reward, default) |
| `diff_sharpe` | differential Sharpe ratio (Moody & Saffell 2001) |
| `drawdown_penalized` | net asset change minus every increase in drawdown |
| `legacy_shaped` *(optional)* | the notebook's shaping: bonus per position change + 1 % of profits |

The agent is the one with the best **mean validation Sharpe** in 06 (never chosen on test), with its tuned
hyperparameters. The reward the final model uses is picked the same way.

In [ ]:
# ---- setup: Drive, code, packages (identical in every notebook) -------------------------
BRANCH = "exp/07-reward-comparison"                                  # the branch this notebook belongs to
REPO = "https://github.com/taimoor-ahmed-1/FinAI-CryptoRaiders.git"
DRIVE_ROOT = "/content/drive/MyDrive/thesis_rl"      # data in, every output out - survives disconnects

import os, sys, subprocess
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/FinAI-CryptoRaiders"
    if not os.path.exists(REPO_DIR):
        # sparse clone: only experiments/ is checked out - the repo's data folders are never downloaded
        subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", "--branch", BRANCH, REPO, REPO_DIR], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "sparse-checkout", "set", "--no-cone", "/experiments/"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", BRANCH, "FETCH_HEAD"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/experiments/requirements-colab.txt"], check=True)
    EXP = f"{REPO_DIR}/experiments"
else:                                                # local run from a checkout: notebooks/ -> experiments/
    EXP = os.path.abspath("..")
    DRIVE_ROOT = os.environ.get("THESIS_RL_ROOT", os.path.abspath("../../thesis_rl_outputs"))
sys.path.insert(0, EXP)
os.chdir(EXP)

import torch
from rltrade import paths, seeding
L = paths.layout(DRIVE_ROOT)
print("code:", BRANCH, seeding.git_commit(EXP), "| device:", "cuda - " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
if not torch.cuda.is_available(): print('WARNING: no GPU - Runtime > Change runtime type > T4 GPU')

In [ ]:
seeding.record_environment(L.report / "environment", "07_reward_comparison")   # exact environment of this run (pip freeze, CUDA, GPU)

In [ ]:
import json, dataclasses, pandas as pd, matplotlib.pyplot as plt
from rltrade.env import EnvConfig
from rltrade.runner import RunSpec, run_seeds
from rltrade import results as R
SEEDS = [0, 1, 2, 3, 4]
REWARDS = ["net_asset_change", "diff_sharpe", "drawdown_penalized"]
INCLUDE_LEGACY_SHAPED = False
TOTAL_STEPS, EVAL_EVERY = 2_000_000, 100_000
F = paths.factor_dir(L)

res = R.load_results(L.runs)
a6 = res[(res.experiment == "06_agents") & (res.agent != "buy_and_hold")]
AGENT = a6.groupby("agent").val_sharpe.mean().idxmax() if len(a6) else "ppo"
tuned = json.loads((L.runs / "tuning" / "best_hp.json").read_text()) if (L.runs / "tuning" / "best_hp.json").exists() else {}
HP = tuned.get(AGENT, {}).get("agent_hp", {})
SCALE = tuned.get(AGENT, {}).get("env", {}).get("reward_scale")   # tuned for equity-based rewards
print("agent (best mean validation Sharpe in 06):", AGENT, HP, "| reward_scale", SCALE)

In [ ]:
for reward in REWARDS + (["legacy_shaped"] if INCLUDE_LEGACY_SHAPED else []):
    # diff_sharpe is scale-free and keeps its own scale; the equity-based rewards use the tuned one
    env = EnvConfig(state="D", reward=reward, reward_scale=None if reward == "diff_sharpe" else SCALE)
    run_seeds(RunSpec("07_rewards", AGENT, env=env, agent_hp=HP, total_steps=TOTAL_STEPS, eval_every=EVAL_EVERY),
              SEEDS, L.prepared, F, L.runs)

## Results

In [ ]:
df = R.load_results(L.runs); df = df[df.experiment == "07_rewards"]
sm = R.summarise(df, metrics=["val_sharpe"] + R.HEADLINE)
table = R.markdown_table(sm, metrics=["val_sharpe"] + R.HEADLINE, group=("reward",))
print(table)
chosen = sm.sort_values("val_sharpe_mean", ascending=False).iloc[0]["reward"]
print("reward chosen on validation:", chosen)
(L.report / "rewards.md").write_text(f"# Reward comparison ({AGENT}, configuration D; test, net of costs)\n\n" + table
                                    + f"\n\nChosen on validation Sharpe: **{chosen}**\n", encoding="utf-8")
groups = {r: df[df.reward == r] for r in df.reward.unique()}
R.plot_curves(df, L.runs, groups, path=L.report / "rewards_equity.png"); plt.show()